In [ ]:
# GERADOR SINTÉTICO DE DADOS - SAC MÓVEIS RESIDENCIAIS
# ==============================================================================
import pandas as pd
import random

templates = {
    'vendas': {
        's': ['', 'Olá', 'Bom dia', 'Gostaria de saber', 'Por favor'],
        'a': ['quero comprar', 'qual o preco do', 'tem cupom para', 'como faco para adquirir', 'desejo orcamento de'],
        'o': ['sofa retratil 3 lugares', 'conjunto de mesa de jantar', 'guarda roupa casal', 'painel para tv', 'colchao queen size']
    },
    'suporte': {
        's': ['', 'Oi', 'Preciso de ajuda', 'Por gentileza', 'Socorro'],
        'a': ['como montar o', 'onde baixo o manual do', 'estou com duvida no', 'veio faltando parafuso no', 'preciso de assistencia para'],
        'o': ['armario de cozinha', 'rack da sala', 'berco do bebe', 'esquema de montagem', 'manual da estante']
    },
    'trocas_devolucoes': {
        's': ['', 'Olá', 'Por favor', 'Gostaria de solicitar', 'Quero abrir'],
        'a': ['preciso trocar o', 'quero devolver a', 'como solicito o estorno do', 'desejo solicitar a troca da', 'como funciona a devolucao do'],
        'o': ['produto com defeito', 'mesa que veio arranhada', 'cadeira no prazo de 7 dias', 'pedido cancelado', 'item com avaria']
    },
    'reclamacoes': {
        's': ['', 'Urgente', 'Pessimo atendimento', 'Absurdo', 'Quero registrar'],
        'a': ['estou indignado com o', 'quero fazer uma queixa do', 'estou reclamando do', 'produto veio quebrado e o', 'atendimento horrivel do'],
        'o': ['atraso na minha entrega', 'servico de montagem', 'sac que nao responde', 'pos venda da loja', 'estado do meu movel']
    },
    'logistica_entregas': {
        's': ['', 'Olá', 'Bom dia', 'Por gentileza', 'Preciso saber'],
        'a': ['onde esta o meu', 'qual o prazo de entrega do', 'como rastreio a', 'qual a transportadora do', 'quando chega o'],
        'o': ['meu pedido', 'codigo de rastreamento', 'movel comprado', 'status do envio', 'agendamento da entrega']
    }
}

amostras = []
random.seed(42)

for intencao, comp in templates.items():
    for _ in range(20):  # Total: 100 amostras (20 por classe)
        s = random.choice(comp['s'])
        a = random.choice(comp['a'])
        o = random.choice(comp['o'])
        frase = f"{s} {a} {o}".strip().capitalize()
        amostras.append({'texto': frase, 'intencao': intencao})

df_moveis = pd.DataFrame(amostras)
df_moveis.to_csv('dataset_moveis_100.csv', index=False, encoding='utf-8')

print(" Dataset 'dataset_moveis_100.csv' criado com 100 frases distribuidas em 5 intencoes!")


 Dataset 'dataset_moveis_100.csv' criado com 100 frases distribuidas em 5 intencoes!


In [ ]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix,f1_score
from sklearn.pipeline import Pipeline



# 1. Carregar dataset do CSV
df = pd.read_csv('dataset_moveis_100.csv')

# 2. Divisão Treino e Teste
X_train, X_test, y_train, y_test = train_test_split(
    df['texto'], df['intencao'], test_size=0.30, random_state=42, stratify=df['intencao']
)

pipeline_tree = Pipeline([
     ('vectorizer', TfidfVectorizer()),
     ('classifier', DecisionTreeClassifier())
 ])

pipeline_tree.fit(X_train,y_train)

LIMIAR_CONFIANCA = 0.50

y_pred = pipeline_tree.predict(X_test)
print(classification_report(y_test, y_pred, zero_division=0))
print(confusion_matrix(y_test, y_pred))

frase = ['Gostaria de saber mais detalhes sobre esse produto e o valor.',
'Esse produto ainda está disponível para compra?',
'Meu pedido chegou com defeito e gostaria de uma solução.',
'Estou muito insatisfeito, pois meu pedido chegou diferente do que solicitei.',
'Gostaria de solicitar a troca do produto, pois o tamanho não ficou adequado.',
'Quero devolver o produto e saber como funciona o processo de reembolso',
'Preciso de ajuda para acessar minha conta, pois não consigo fazer login.',
'Gostaria de saber onde está meu pedido e qual é a previsão de entrega' ]

probs = pipeline_tree.predict_proba(frase)
maior_prob = np.max(probs)
intencao = pipeline_tree.predict(frase)[0]


acuracia = accuracy_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred, average='weighted')
print("Acurácia:", acuracia)
print("F1-Score Weighted:", f1)

if maior_prob >= LIMIAR_CONFIANCA:
  print(f"Intenção prevista é: ",{intencao})
  print(f"A probabilidade é: {maior_prob  * 100:.2f}%")
else:
  print("Desculpe, não entendi sua solicitação. Encaminhando você para um atendente humano...")
pass

                    precision    recall  f1-score   support

logistica_entregas       0.80      0.67      0.73         6
       reclamacoes       1.00      0.33      0.50         6
           suporte       1.00      1.00      1.00         6
 trocas_devolucoes       0.60      1.00      0.75         6
            vendas       0.71      0.83      0.77         6

          accuracy                           0.77        30
         macro avg       0.82      0.77      0.75        30
      weighted avg       0.82      0.77      0.75        30

[[4 0 0 0 2]
 [1 2 0 3 0]
 [0 0 6 0 0]
 [0 0 0 6 0]
 [0 0 0 1 5]]
Acurácia: 0.7666666666666667
F1-Score Weighted: 0.7493006993006993
Intenção prevista é:  {'vendas'}
A probabilidade é: 100.00%
